<div style="text-align: center;">
  <span style="color: lightskyblue; font-weight: bold; font-size: 50px;">
    Test File for Depth Camera (D435i)
  </span>
</div>

<span style="color: magenta; font-weight: bold; font-size: 30px;">
Coordinate Information
</span>

<span style="color: white; font-weight: normal; font-size: 20px;">
pixel: (u,v), origin = top-left point, u: left-to-right, v: top-to-bottom
<br>
depth: z
<br>
3D position: (X,Y,Z), origin = top-left point, X: left-to-right, Y: top-to-bottom, Z=z(depth)
</span>

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Check recognition of D435i
</span>

In [2]:
ctx = rs.context()
devices = ctx.query_devices()

print("Number of RealSense devices:", len(devices))

for dev in devices:
    print("Name:", dev.get_info(rs.camera_info.name))
    print("Serial:", dev.get_info(rs.camera_info.serial_number))
    print("Firmware:", dev.get_info(rs.camera_info.firmware_version))

Number of RealSense devices: 1
Name: Intel RealSense D435I
Serial: 250122070658
Firmware: 5.15.1.55


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Robot
</span>

In [2]:
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot
robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Reset the Control Manager!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************





<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Power OFF
</span>

In [4]:
ysw.powerOFF(robot, robotConfig)

****************************** [powerOFF Starts] *******************************

[INFO] SUCCESS: Disable Grippers!
[INFO] SUCCESS: Disable the Control Manager!

****************************** [PowerOFF is Done] ******************************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Posture Change
</span>

In [3]:
Desired_Pos = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, Desired_Pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Vision
</span>

In [3]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640,480), fps=30)
d435i.initialize()

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Simple Test of D435i
</span>

In [5]:
# Simple test of RGB-D camera
# Click ESC: Terminate
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()
        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)
        
        if (d435i.rgb_on):
            d435i.imshow("color_image", color_image)
        if (d435i.depth_on):
            d435i.imshow("depth_image", depth_colorMap)
        
        key = d435i.waitKey(1)
        if key == 27:       # ESC key
            break
        
finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()
        

[INFO] D435i pipeline started


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Distance of center pixel
</span>

In [6]:
# Get distance value of center pixel of image
# Click ESC: Terminate
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()
        
        if color_frame is None or depth_frame is None:
            continue
        
        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top

        distance = d435i.getDepth(depth_frame, (u, v))   # unit = [m]

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,                   
            f"Depth: {distance:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of center pixel based on Camera Reference Frame
</span>

In [7]:
# Show center 3D point position [x,y,z] in camera frame
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top 

        pos = d435i.get3DPos(depth_frame, (u,v))
        X, Y, Z = pos

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,
            f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of clicked point 
</span>

In [8]:
d435i.openWindowWithMouse("color_image")

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        if d435i.clicked_point is not None:
            u, v = d435i.clicked_point

            X, Y, Z = d435i.get3DPos(depth_frame, (u,v))

            cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
            cv2.putText(
                color_image,
                f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            print(f"Pixel=({u}, {v}), Camera 3D=({X:.3f}, {Y:.3f}, {Z:.3f}) m")

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started
Clicked pixel: u=364, v=271
Pixel=(364, 271), Camera 3D=(0.080, 0.036, 0.889) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.895) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.891) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.899) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.893) m
Pixel=(364, 271), Camera 3D=(0.080, 0.036, 0.884) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.894) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.894) m
Pixel=(364, 271), Camera 3D=(0.080, 0.036, 0.886) m
Pixel=(364, 271), Camera 3D=(0.080, 0.036, 0.889) m
Pixel=(364, 271), Camera 3D=(0.080, 0.036, 0.889) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.890) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.899) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.897) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.890) m
Pixel=(364, 271), Camera 3D=(0.081, 0.036, 0.898) m
Pixel=(364, 271), Camera 3D=(0.080, 0.036, 0.885) m
Clicked pixel: u=486, v=145
Pixel=(486, 145), Camera 3D=(0

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Center 3D position of Colored Object (2D BBox)
</span>

In [9]:
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = d435i.detectColorObjectContour(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,
            minimum_area=300                    # Can be changed
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"pixel={obj['pixel']}, "
                f"pos={np.round(obj['pos'], 3)}, "
                f"area={obj['area']:.1f}"
            )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)
        d435i.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started
[object 0] pixel=[466, 386], pos=[0.187 0.166 0.722], area=1221.0
[object 1] pixel=[183, 352], pos=[-0.162  0.136  0.782], area=301.0
[object 2] pixel=[490, 253], pos=[0.273 0.01  0.915], area=300.5
[object 5] pixel=[257, 186], pos=[-0.087 -0.101  1.013], area=13410.5
[object 7] pixel=[438, 50], pos=[ 0.399 -0.607  1.878], area=685.5
[object 0] pixel=[466, 388], pos=[0.188 0.169 0.726], area=1164.0
[object 2] pixel=[490, 237], pos=[ 0.265 -0.014  0.891], area=648.0
[object 3] pixel=[257, 187], pos=[-0.086 -0.099  1.011], area=13393.0
[object 4] pixel=[438, 53], pos=[ 0.396 -0.594  1.867], area=753.0
[object 0] pixel=[466, 388], pos=[0.188 0.169 0.726], area=1195.0
[object 1] pixel=[181, 354], pos=[-0.164  0.138  0.781], area=321.0
[object 2] pixel=[490, 238], pos=[ 0.265 -0.012  0.891], area=654.5
[object 3] pixel=[257, 187], pos=[-0.086 -0.099  1.011], area=13401.5
[object 5] pixel=[438, 52], pos=[ 0.396 -0.597  1.867], area=745.0
[object 0] pixel=[466, 3

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Center 3D position of Colored Object (3D BBox)
</span>

In [ ]:
d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,       # Can be changed
            minimum_area=300                    # Can be changed
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
            )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)
        d435i.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started
[object 0] center_3d=[-0.237  0.151  0.727], bbox_3d_size=[0.039 0.053 0.053]
[object 1] center_3d=[0.226 0.133 0.756], bbox_3d_size=[0.064 0.044 0.039]
[object 2] center_3d=[0.263 0.005 0.907], bbox_3d_size=[0.044 0.039 0.032]
[object 3] center_3d=[-0.02   0.116  0.786], bbox_3d_size=[0.153 0.243 0.231]
[object 4] center_3d=[ 0.254 -0.037  0.851], bbox_3d_size=[0.011 0.018 0.026]
[object 5] center_3d=[-0.741 -0.65   1.796], bbox_3d_size=[0.018 0.012 0.01 ]
[object 6] center_3d=[ 0.39  -0.602  1.884], bbox_3d_size=[0.046 0.188 0.151]


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

[object 0] center_3d=[-0.24   0.154  0.733], bbox_3d_size=[0.044 0.055 0.066]
[object 1] center_3d=[0.227 0.135 0.757], bbox_3d_size=[0.064 0.042 0.036]
[object 2] center_3d=[-0.017  0.122  0.785], bbox_3d_size=[0.167 0.247 0.227]
[object 3] center_3d=[ 0.263 -0.008  0.893], bbox_3d_size=[0.05  0.07  0.084]
[object 4] center_3d=[ 0.394 -0.585  1.878], bbox_3d_size=[0.048 0.196 0.156]
[object 0] center_3d=[-0.238  0.153  0.73 ], bbox_3d_size=[0.045 0.051 0.054]
[object 1] center_3d=[0.228 0.136 0.76 ], bbox_3d_size=[0.066 0.043 0.032]
[object 2] center_3d=[-0.017  0.124  0.784], bbox_3d_size=[0.166 0.246 0.222]
[object 3] center_3d=[ 0.263 -0.008  0.898], bbox_3d_size=[0.048 0.07  0.08 ]
[object 4] center_3d=[ 0.393 -0.589  1.884], bbox_3d_size=[0.048 0.193 0.17 ]
[object 0] center_3d=[-0.24   0.152  0.726], bbox_3d_size=[0.048 0.057 0.047]
[object 1] center_3d=[0.228 0.134 0.759], bbox_3d_size=[0.065 0.045 0.041]
[object 2] center_3d=[-0.018  0.122  0.783], bbox_3d_size=[0.168 0.246 0.